## 类别解码指标分析（NC vs MCI vs AD）

从各模型 `results.txt` 中读取每个 ITR 的 **Subject-level** `Confusion Matrix (Count)`，按 `[NC, MCI, AD]` 顺序分别计算三类的 Precision / Recall / F1。

- **数据集：** CAEEG、CAUEEG（各 9 个模型，ITR 1–5）
- **粒度：** Subject-level 计数混淆矩阵；平均结果为 5 个 run 指标的均值 ± 标准差（`ddof=0`）
- **一类 vs 其余：**
  - Precision = TP / (TP + FP)
  - Recall = TP / (TP + FN)
  - F1 = 2 · Precision · Recall / (Precision + Recall)
- **表格分组：** 数据集 → 类别（NC / MCI / AD）→ 模型
- **输出：** `figs/CAEEG_CAUEEG_decoding_metrics.xlsx`（`ReadMe`、`Paper_NCvsMCIvsAD`、`Paper_NCvsMCI`、`Paper_All`、`Per_run`）


In [4]:
import math
import re
from pathlib import Path

import numpy as np
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "run.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
RESULTS_ROOT = PROJECT_ROOT / "results"
SAVE_DIR = PROJECT_ROOT / "figs"
SAVE_PATH = SAVE_DIR / "CAEEG_CAUEEG_decoding_metrics.xlsx"

CLASS_ORDER = ["NC", "MCI", "AD"]
N_ITR = 5
STD_DDOF = 0  # 与 results.txt 中 Mean ± Std 的 np.std 默认一致
DATASET_ORDER = ["CAEEG", "CAUEEG"]
TASK_ORDER = ["NCvsMCIvsAD", "NCvsMCI"]
TASK_CLASSES = {
    "NCvsMCIvsAD": ["NC", "MCI", "AD"],
    "NCvsMCI": ["NC", "MCI"],
}
MODEL_ORDER = [
    "EEGNet",
    "Conformer",
    "ADformer",
    "Medformer",
    "TSLANet",
    "MedGNN",
    "STCGRU",
    "CodimNet",
    "ADSSTNet",
]

DATASETS = {
    "CAEEG": {
        "root": RESULTS_ROOT,
        "folders": [
            ("EEGNet", "CAEEG3_1s_00_EO_Aug-EEGNet-NCvsMCIvsAD-noAugmentation"),
            ("Conformer", "CAEEG3_1s_00_EO_Aug-Conformer-NCvsMCIvsAD-noAugmentation"),
            ("ADformer", "CAEEG3_1s_00_EO_Aug-ADformer-NCvsMCIvsAD-noAugmentation"),
            ("Medformer", "CAEEG3_1s_00_EO_Aug-Medformer-NCvsMCIvsAD-noAugmentation"),
            ("TSLANet", "CAEEG3_1s_00_EO_Aug-TSLANet-NCvsMCIvsAD-noAugmentation"),
            ("MedGNN", "CAEEG3_1s_00_EO_Aug-MedGNN-NCvsMCIvsAD-noAugmentation"),
            ("STCGRU", "CAEEG3_1s_00_EO_Aug-STCGRU-NCvsMCIvsAD-noAugmentation"),
            ("CodimNet", "CAEEG3_1s_00_EO_Aug-CodimNet-NCvsMCIvsAD-noAugmentation"),
            ("ADSSTNet", "CAEEG3_1s_00_EO_Aug-ADSSTNet-NCvsMCIvsAD-oneFreBanDAGnoRanCrop"),
        ],
    },
    "CAUEEG": {
        "root": RESULTS_ROOT,
        "folders": [
            ("EEGNet", "CAUEEG_Dementia_1s_00_EO_Aug-EEGNet-NCvsMCIvsAD-noAugmentation"),
            ("Conformer", "CAUEEG_Dementia_1s_00_EO_Aug-Conformer-NCvsMCIvsAD-noAugmentation"),
            ("ADformer", "CAUEEG_Dementia_1s_00_EO_Aug-ADformer-NCvsMCIvsAD-noAugmentation"),
            ("Medformer", "CAUEEG_Dementia_1s_00_EO_Aug-Medformer-NCvsMCIvsAD-noAugmentation"),
            ("TSLANet", "CAUEEG_Dementia_1s_00_EO_Aug-TSLANet-NCvsMCIvsAD-noAugmentation"),
            ("MedGNN", "CAUEEG_Dementia_1s_00_EO_Aug-MedGNN-NCvsMCIvsAD-noAugmentation"),
            ("STCGRU", "CAUEEG_Dementia_1s_00_EO_Aug-STCGRU-NCvsMCIvsAD-noAugmentation"),
            ("CodimNet", "CAUEEG_Dementia_1s_00_EO_Aug-CodimNet-NCvsMCIvsAD-noAugmentation"),
            ("ADSSTNet", "CAUEEG_Dementia_1s_00_EO_Aug-ADSSTNet-NCvsMCIvsAD-oneFreBanDAGnoRanCrop"),
        ],
    },
}

print("SAVE_PATH =", SAVE_PATH)
print("TASK_ORDER =", TASK_ORDER)
for _name in DATASET_ORDER:
    _root = DATASETS[_name]["root"]
    print(f"{_name}: root exists={_root.exists()}  n_models={len(DATASETS[_name]['folders'])}")


SAVE_PATH = /root/autodl-tmp/ADSST-Net/figs/CAEEG_CAUEEG_decoding_metrics.xlsx
TASK_ORDER = ['NCvsMCIvsAD', 'NCvsMCI']
CAEEG: root exists=True  n_models=9
CAUEEG: root exists=True  n_models=9


In [5]:
def _parse_matrix_line(line: str) -> list[float]:
    parts = re.split(r"\s+", line.strip())
    values = []
    for p in parts:
        if p:
            values.append(float(p))
    return values


def _read_count_matrix(lines: list[str], start: int) -> np.ndarray:
    """从 Confusion Matrix (Count) 标题的下一行起读取方阵。"""
    rows = []
    i = start
    while i < len(lines):
        stripped = lines[i].strip()
        if not stripped:
            break
        if stripped.startswith("Confusion Matrix") or stripped.startswith("ITR "):
            break
        if stripped.endswith(":") and not re.match(r"^[\d.\t %+-]+", stripped):
            break
        try:
            rows.append(_parse_matrix_line(lines[i]))
        except ValueError:
            break
        i += 1
        if rows and len(rows) == len(rows[0]):
            break
    if not rows:
        raise ValueError(f"未在第 {start + 1} 行附近解析到计数混淆矩阵")
    cm = np.array(rows, dtype=float)
    if cm.ndim != 2 or cm.shape[0] != cm.shape[1]:
        raise ValueError(f"混淆矩阵不是方阵: shape={cm.shape}")
    return cm


LEVEL_HEADERS = {
    "Sample-level": "Sample-level results",
    "Subject-level": "Subject-level results",
}


def parse_itr_count_cms(path: Path) -> dict[str, dict[int, np.ndarray]]:
    """解析每个 ITR 的 Sample-level / Subject-level Confusion Matrix (Count)。"""
    lines = path.read_text(encoding="utf-8").splitlines()
    avg_start = next((i for i, ln in enumerate(lines) if "所有ITR的平均结果" in ln), len(lines))
    itr_positions = [i for i, ln in enumerate(lines[:avg_start]) if re.match(r"ITR\s+\d+\s+结果", ln)]
    if not itr_positions:
        raise ValueError(f"未找到 ITR 结果块: {path}")

    out = {level: {} for level in LEVEL_HEADERS}
    for idx, pos in enumerate(itr_positions):
        itr_num = int(re.search(r"ITR\s+(\d+)", lines[pos]).group(1))
        block_end = itr_positions[idx + 1] if idx + 1 < len(itr_positions) else avg_start
        sample_start = next(
            (i for i in range(pos, block_end) if "Sample-level results" in lines[i]),
            None,
        )
        subj_start = next(
            (i for i in range(pos, block_end) if "Subject-level results" in lines[i]),
            None,
        )
        ranges = {
            "Sample-level": (sample_start, subj_start if subj_start is not None else block_end),
            "Subject-level": (subj_start, block_end),
        }
        for level, (start, end) in ranges.items():
            if start is None:
                raise ValueError(f"{path.name}: ITR {itr_num} 缺少 {level} results")
            cm_idx = next(
                (i for i in range(start, end) if "Confusion Matrix (Count)" in lines[i]),
                None,
            )
            if cm_idx is None:
                raise ValueError(f"{path.name}: ITR {itr_num} 缺少 {level} Confusion Matrix (Count)")
            out[level][itr_num] = _read_count_matrix(lines, cm_idx + 1)
    return out


def safe_div(num: float, den: float) -> float:
    return float(num) / float(den) if den else 0.0


def resolve_results_txt(root: Path, folder_3class: str, model: str, task: str) -> Path:
    """三分类文件夹替换 NCvsMCIvsAD -> 目标任务名；若不存在则按模型名回退搜索。"""
    candidate = folder_3class if task == "NCvsMCIvsAD" else folder_3class.replace("NCvsMCIvsAD", task)
    txt = root / candidate / "results.txt"
    if txt.exists():
        return txt
    dataset_prefix = folder_3class.split("-", 1)[0]
    hits = []
    for p in root.iterdir():
        if not p.is_dir():
            continue
        name = p.name
        if not name.startswith(dataset_prefix):
            continue
        if f"-{model}-" not in name:
            continue
        if task not in name:
            continue
        if task == "NCvsMCI" and "NCvsMCIvsAD" in name:
            continue
        if (p / "results.txt").exists():
            hits.append(p / "results.txt")
    if len(hits) == 1:
        print(f"路径回退: {model} / {task} -> {hits[0].parent.name}")
        return hits[0]
    detail = "；".join(h.parent.name for h in hits) if hits else "无匹配"
    raise FileNotFoundError(f"找不到 results.txt: {txt}（回退匹配: {detail}）")


def class_metrics_from_cm(cm: np.ndarray, class_index: int, class_names: list[str]) -> dict[str, float]:
    """由混淆矩阵计算指定类别的 Precision / Recall / F1（百分数）。"""
    n = len(class_names)
    if cm.shape != (n, n):
        raise ValueError(f"期望 {n}x{n} 矩阵（{class_names}），实际 {cm.shape}")
    tp = float(cm[class_index, class_index])
    fp = float(cm[:, class_index].sum() - tp)
    fn = float(cm[class_index, :].sum() - tp)
    support = float(cm[class_index, :].sum())
    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    f1 = safe_div(2 * precision * recall, precision + recall)
    return {
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "Support": support,
        "Precision": precision * 100.0,
        "Recall": recall * 100.0,
        "F1": f1 * 100.0,
    }


def format_mean_std(mean: float, std: float, digits: int = 2) -> str:
    return f"{mean:.{digits}f} ± {std:.{digits}f}"


print("parser ready")


parser ready


In [6]:
per_run_rows = []
missing = []

for task in TASK_ORDER:
    class_names = TASK_CLASSES[task]
    for dataset in DATASET_ORDER:
        cfg = DATASETS[dataset]
        for model, folder in cfg["folders"]:
            try:
                txt_path = resolve_results_txt(cfg["root"], folder, model, task)
            except FileNotFoundError as err:
                missing.append(str(err))
                continue
            cms = parse_itr_count_cms(txt_path)["Subject-level"]
            for itr in range(1, N_ITR + 1):
                if itr not in cms:
                    raise KeyError(f"{task} / {dataset} / {model}: 缺少 ITR {itr}")
                cm = cms[itr]
                for class_name in class_names:
                    metrics = class_metrics_from_cm(cm, class_names.index(class_name), class_names)
                    per_run_rows.append(
                        {
                            "Task": task,
                            "Dataset": dataset,
                            "Class": class_name,
                            "Model": model,
                            "ITR": itr,
                            "Precision (%)": metrics["Precision"],
                            "Recall (%)": metrics["Recall"],
                            "F1 Score (%)": metrics["F1"],
                            "TP": metrics["TP"],
                            "FP": metrics["FP"],
                            "FN": metrics["FN"],
                            "Support": metrics["Support"],
                            "folder": txt_path.parent.name,
                        }
                    )

if missing:
    raise FileNotFoundError("缺少 results.txt:\n" + "\n".join(missing))

per_run_df = pd.DataFrame(per_run_rows)
per_run_df["Task"] = pd.Categorical(per_run_df["Task"], TASK_ORDER, ordered=True)
per_run_df["Dataset"] = pd.Categorical(per_run_df["Dataset"], DATASET_ORDER, ordered=True)
per_run_df["Class"] = pd.Categorical(per_run_df["Class"], CLASS_ORDER, ordered=True)
per_run_df["Model"] = pd.Categorical(per_run_df["Model"], MODEL_ORDER, ordered=True)
per_run_df = per_run_df.sort_values(["Task", "Dataset", "Class", "Model", "ITR"]).reset_index(drop=True)
per_run_df = per_run_df[
    ["Task", "Dataset", "Class", "Model", "ITR", "Precision (%)", "Recall (%)", "F1 Score (%)", "TP", "FP", "FN", "Support", "folder"]
]

avg_rows = []
for (task, dataset, class_name, model), sub in per_run_df.groupby(
    ["Task", "Dataset", "Class", "Model"], observed=True, sort=True
):
    prec = sub["Precision (%)"].to_numpy(dtype=float)
    rec = sub["Recall (%)"].to_numpy(dtype=float)
    f1 = sub["F1 Score (%)"].to_numpy(dtype=float)
    avg_rows.append(
        {
            "Task": task,
            "Dataset": dataset,
            "Class": class_name,
            "Model": model,
            "Precision_mean": float(np.mean(prec)),
            "Precision_std": float(np.std(prec, ddof=STD_DDOF)),
            "Recall_mean": float(np.mean(rec)),
            "Recall_std": float(np.std(rec, ddof=STD_DDOF)),
            "F1_mean": float(np.mean(f1)),
            "F1_std": float(np.std(f1, ddof=STD_DDOF)),
            **{f"Precision_ITR{i}": float(prec[i - 1]) for i in range(1, N_ITR + 1)},
            **{f"Recall_ITR{i}": float(rec[i - 1]) for i in range(1, N_ITR + 1)},
            **{f"F1_ITR{i}": float(f1[i - 1]) for i in range(1, N_ITR + 1)},
        }
    )

avg_df = pd.DataFrame(avg_rows)
avg_df["Task"] = pd.Categorical(avg_df["Task"], TASK_ORDER, ordered=True)
avg_df["Dataset"] = pd.Categorical(avg_df["Dataset"], DATASET_ORDER, ordered=True)
avg_df["Class"] = pd.Categorical(avg_df["Class"], CLASS_ORDER, ordered=True)
avg_df["Model"] = pd.Categorical(avg_df["Model"], MODEL_ORDER, ordered=True)
avg_df = avg_df.sort_values(["Task", "Dataset", "Class", "Model"]).reset_index(drop=True)
avg_df["Precision (%)"] = [
    format_mean_std(m, s) for m, s in zip(avg_df["Precision_mean"], avg_df["Precision_std"])
]
avg_df["Recall (%)"] = [
    format_mean_std(m, s) for m, s in zip(avg_df["Recall_mean"], avg_df["Recall_std"])
]
avg_df["F1 Score (%)"] = [
    format_mean_std(m, s) for m, s in zip(avg_df["F1_mean"], avg_df["F1_std"])
]
avg_df["LaTeX"] = [
    f"{task} & {dataset} & {cls} & {model} & {p} & {r} & {f} \\\\"
    for task, dataset, cls, model, p, r, f in zip(
        avg_df["Task"],
        avg_df["Dataset"],
        avg_df["Class"],
        avg_df["Model"],
        avg_df["Precision (%)"],
        avg_df["Recall (%)"],
        avg_df["F1 Score (%)"],
    )
]

n_binary_avg = int((avg_df["Task"] == "NCvsMCI").sum())
n_ternary_avg = int((avg_df["Task"] == "NCvsMCIvsAD").sum())
n_binary_run = int((per_run_df["Task"] == "NCvsMCI").sum())
n_ternary_run = int((per_run_df["Task"] == "NCvsMCIvsAD").sum())
if n_binary_avg != 36 or n_binary_run != 180:
    raise RuntimeError(
        f"二分类结果行数不对: Paper {n_binary_avg} (期望 36), Per_run {n_binary_run} (期望 180)"
    )
print(
    f"已解析 {len(TASK_ORDER)} 个任务 × {len(DATASET_ORDER)} 个数据集 × {len(MODEL_ORDER)} 个模型 × {N_ITR} 个 run（Subject-level）"
)
print(f"NCvsMCIvsAD: Paper {n_ternary_avg} 行, Per_run {n_ternary_run} 行")
print(f"NCvsMCI    : Paper {n_binary_avg} 行, Per_run {n_binary_run} 行")
display_cols = ["Task", "Dataset", "Class", "Model", "Precision (%)", "Recall (%)", "F1 Score (%)"]
print("\n===== 5-run 平均（按任务→数据集→类别分组）=====")
print(avg_df[display_cols].to_string(index=False))
print("\n===== 各 run（Subject-level，按任务→数据集→类别分组）=====")
print(per_run_df[["Task", "Dataset", "Class", "Model", "ITR", "Precision (%)", "Recall (%)", "F1 Score (%)"]].to_string(index=False))


已解析 2 个任务 × 2 个数据集 × 9 个模型 × 5 个 run（Subject-level）
NCvsMCIvsAD: Paper 54 行, Per_run 270 行
NCvsMCI    : Paper 36 行, Per_run 180 行

===== 5-run 平均（按任务→数据集→类别分组）=====
       Task Dataset Class     Model Precision (%)    Recall (%)  F1 Score (%)
NCvsMCIvsAD   CAEEG    NC    EEGNet 69.16 ± 15.50 65.00 ± 24.38 63.44 ± 14.59
NCvsMCIvsAD   CAEEG    NC Conformer 77.27 ± 13.02 65.00 ± 26.56 64.67 ± 19.32
NCvsMCIvsAD   CAEEG    NC  ADformer 81.11 ± 10.48 61.67 ± 22.73 65.91 ± 14.60
NCvsMCIvsAD   CAEEG    NC Medformer 73.33 ± 14.09 58.33 ± 29.81 58.00 ± 17.88
NCvsMCIvsAD   CAEEG    NC   TSLANet  70.95 ± 6.46 58.33 ± 18.26 62.75 ± 13.36
NCvsMCIvsAD   CAEEG    NC    MedGNN 74.31 ± 13.04 68.33 ± 33.08 63.13 ± 19.95
NCvsMCIvsAD   CAEEG    NC    STCGRU  57.28 ± 4.96  93.33 ± 9.72  70.84 ± 5.72
NCvsMCIvsAD   CAEEG    NC  CodimNet  65.89 ± 6.88  85.00 ± 6.24  74.04 ± 5.68
NCvsMCIvsAD   CAEEG    NC  ADSSTNet  70.88 ± 8.19  90.00 ± 8.16  78.97 ± 6.37
NCvsMCIvsAD   CAEEG   MCI    EEGNet 28.98 ± 20.38 31.11

In [7]:
THIN = Border(
    left=Side(style="thin", color="B0B0B0"),
    right=Side(style="thin", color="B0B0B0"),
    top=Side(style="thin", color="B0B0B0"),
    bottom=Side(style="thin", color="B0B0B0"),
)
HEADER_FILL = PatternFill("solid", fgColor="1F4E79")
HEADER_FONT = Font(name="Calibri", bold=True, color="FFFFFF", size=11)
TITLE_FONT = Font(name="Calibri", bold=True, size=13, color="1F4E79")
PROPOSED_FILL = PatternFill("solid", fgColor="FFF2CC")
CLASS_FILLS = {
    "NC": PatternFill("solid", fgColor="D6EAF8"),
    "MCI": PatternFill("solid", fgColor="D5F5E3"),
    "AD": PatternFill("solid", fgColor="FADBD8"),
}
DATASET_FILLS = {
    "CAEEG": PatternFill("solid", fgColor="E8E8E8"),
    "CAUEEG": PatternFill("solid", fgColor="F5E6CC"),
}
TASK_FILLS = {
    "NCvsMCIvsAD": PatternFill("solid", fgColor="D6EAF8"),
    "NCvsMCI": PatternFill("solid", fgColor="E8DAEF"),
}
NOTE_FONT = Font(name="Calibri", italic=True, size=10, color="595959")
CENTER = Alignment(horizontal="center", vertical="center", wrap_text=True)
LEFT = Alignment(horizontal="left", vertical="center", wrap_text=True)


def _style_header(ws, row: int, start_col: int, end_col: int):
    for col in range(start_col, end_col + 1):
        cell = ws.cell(row, col)
        cell.fill = HEADER_FILL
        cell.font = HEADER_FONT
        cell.alignment = CENTER
        cell.border = THIN


def _autosize(ws, min_width=12, max_width=42):
    for col in range(1, ws.max_column + 1):
        width = min_width
        for row in range(1, min(ws.max_row, 80) + 1):
            val = ws.cell(row, col).value
            if val is None:
                continue
            width = max(width, min(max_width, len(str(val)) + 2))
        ws.column_dimensions[get_column_letter(col)].width = width


def _merge_consecutive_column(ws, df, col_name, header_row, fills=None):
    if col_name not in df.columns:
        return
    col_idx = list(df.columns).index(col_name) + 1
    values = [str(v) for v in df[col_name].tolist()]
    start_data = header_row + 1
    i = 0
    while i < len(values):
        j = i + 1
        while j < len(values) and values[j] == values[i]:
            j += 1
        fill = (fills or {}).get(values[i])
        for r in range(start_data + i, start_data + j):
            cell = ws.cell(r, col_idx)
            if fill is not None:
                cell.fill = fill
        if j - i > 1:
            ws.merge_cells(
                start_row=start_data + i,
                start_column=col_idx,
                end_row=start_data + j - 1,
                end_column=col_idx,
            )
            merged = ws.cell(start_data + i, col_idx)
            merged.alignment = CENTER
            merged.border = THIN
        i = j


def _write_df(ws, df: pd.DataFrame, title: str, highlight_model="ADSSTNet"):
    ws["A1"] = title
    ws["A1"].font = TITLE_FONT
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=max(1, len(df.columns)))
    header_row = 3
    for j, col in enumerate(df.columns, start=1):
        ws.cell(header_row, j, col)
    _style_header(ws, header_row, 1, len(df.columns))
    for i, row in enumerate(df.itertuples(index=False), start=header_row + 1):
        values = list(row)
        model_name = values[list(df.columns).index("Model")] if "Model" in df.columns else None
        for j, (col, val) in enumerate(zip(df.columns, values), start=1):
            cell = ws.cell(i, j)
            if isinstance(val, (float, np.floating)) and not (isinstance(val, float) and math.isnan(val)):
                cell.value = float(val)
                if col in {"ITR", "TP", "FP", "FN", "Support"} or str(col).startswith("true"):
                    cell.number_format = "0"
                else:
                    cell.number_format = "0.00"
            else:
                cell.value = None if (isinstance(val, float) and math.isnan(val)) else val
            cell.alignment = LEFT if col in {"folder", "LaTeX"} else CENTER
            cell.border = THIN
            if model_name == highlight_model:
                cell.fill = PROPOSED_FILL
    _merge_consecutive_column(ws, df, "Task", header_row, TASK_FILLS)
    _merge_consecutive_column(ws, df, "Dataset", header_row, DATASET_FILLS)
    _merge_consecutive_column(ws, df, "Class", header_row, CLASS_FILLS)
    _autosize(ws)
    ws.freeze_panes = "A4"
    ws.auto_filter.ref = f"A3:{get_column_letter(len(df.columns))}{3 + len(df)}"


def write_readme(wb: Workbook):
    ws = wb.create_sheet("ReadMe", 0)
    ws["A1"] = "CAEEG / CAUEEG：Subject-level 三分类与二分类 Precision / Recall / F1"
    ws["A1"].font = TITLE_FONT
    notes = [
        "1. 数据来自 CAEEG 与 CAUEEG 各模型 results.txt 中 ITR 1-5 的 Subject-level Confusion Matrix (Count)。",
        "2. 工作表 Paper_NCvsMCIvsAD 为三分类 [NC, MCI, AD]；Paper_NCvsMCI 为二分类 [NC, MCI]。",
        "3. 二分类文件夹由三分类名中的 NCvsMCIvsAD 替换为 NCvsMCI；若路径不存在则按模型名回退匹配。",
        "4. Precision = TP/(TP+FP)；Recall = TP/(TP+FN)；F1 = 2PR/(P+R)。分母为 0 时记为 0。",
        "5. 表格按 任务 → 数据集 → 类别 → 模型 分组。Per_run 给出每个 ITR 的百分数。",
        "6. ADSSTNet 行以黄色底纹标出。",
    ]
    for i, text in enumerate(notes, start=3):
        ws[f"A{i}"] = text
        ws[f"A{i}"].font = NOTE_FONT if i >= 5 else Font(name="Calibri", size=11)
        ws[f"A{i}"].alignment = LEFT
    ws.column_dimensions["A"].width = 130
    ws.row_dimensions[1].height = 22


def save_workbook(wb: Workbook, paths) -> list:
    required = ["ReadMe", "Paper_NCvsMCIvsAD", "Paper_NCvsMCI", "Paper_All", "Per_run"]
    missing_sheets = [s for s in required if s not in wb.sheetnames]
    if missing_sheets:
        raise RuntimeError(f"工作簿缺少工作表: {missing_sheets}；实际: {wb.sheetnames}")
    n_bin = wb["Paper_NCvsMCI"].max_row - 3
    if n_bin < 36:
        raise RuntimeError(f"Paper_NCvsMCI 只有 {n_bin} 行数据，期望 36")
    if "Paper_Subject" in wb.sheetnames:
        raise RuntimeError("不应再写入 Paper_Subject，请重新运行本格")

    saved = []
    for path in paths:
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        dest = path
        try:
            if dest.exists():
                dest.unlink()
        except PermissionError:
            dest = path.with_name(path.stem + "_NEW.xlsx")
            print(f"{path.name} 正被 Excel 占用，改存为 {dest.name}")
        wb.save(dest)
        print(f"已保存: {dest}")
        saved.append(dest)

    from openpyxl import load_workbook as _load_wb
    check = _load_wb(saved[0], read_only=True)
    print("回读工作表:", check.sheetnames)
    if "Paper_NCvsMCI" not in check.sheetnames:
        check.close()
        raise RuntimeError(f"回读失败，{saved[0]} 没有 Paper_NCvsMCI")
    check.close()
    return saved


SAVE_DIR.mkdir(parents=True, exist_ok=True)
if "Task" not in avg_df.columns:
    raise RuntimeError("avg_df 没有 Task 列，请先重新运行上一格（必须包含 NCvsMCI）")

wb = Workbook()
default_ws = wb.active
wb.remove(default_ws)
write_readme(wb)

paper_cols = ["Task", "Dataset", "Class", "Model", "Precision (%)", "Recall (%)", "F1 Score (%)", "LaTeX"]
paper_cols_task = ["Dataset", "Class", "Model", "Precision (%)", "Recall (%)", "F1 Score (%)", "LaTeX"]
per_run_export = per_run_df.drop(columns=["folder"]).copy()

avg_3class = avg_df.loc[avg_df["Task"] == "NCvsMCIvsAD", paper_cols_task].copy()
avg_binary = avg_df.loc[avg_df["Task"] == "NCvsMCI", paper_cols_task].copy()
if len(avg_binary) != 36:
    raise RuntimeError(f"avg_df 中 NCvsMCI 为 {len(avg_binary)} 行，期望 36。请先重新运行上一格。")

ws_3class = wb.create_sheet("Paper_NCvsMCIvsAD")
_write_df(
    ws_3class,
    avg_3class,
    "NCvsMCIvsAD (3-class) Subject-level Precision / Recall / F1  (mean ± std, 5 runs)",
)

ws_binary = wb.create_sheet("Paper_NCvsMCI")
_write_df(
    ws_binary,
    avg_binary,
    "NCvsMCI (binary) Subject-level Precision / Recall / F1  (mean ± std, 5 runs)",
)

ws_all = wb.create_sheet("Paper_All")
_write_df(
    ws_all,
    avg_df[paper_cols].copy(),
    "All tasks: NCvsMCIvsAD + NCvsMCI  (mean ± std, 5 runs)",
)

ws_run = wb.create_sheet("Per_run")
_write_df(ws_run, per_run_export, "Subject-level metrics for each task / dataset / class / model / ITR")

saved = save_workbook(wb, [SAVE_PATH])
print("工作表:", wb.sheetnames)
print("Paper_NCvsMCI 行数:", len(avg_binary), " Paper_NCvsMCIvsAD 行数:", len(avg_3class))
print("最终文件:", saved[0])


已保存: /root/autodl-tmp/ADSST-Net/figs/CAEEG_CAUEEG_decoding_metrics.xlsx
回读工作表: ['ReadMe', 'Paper_NCvsMCIvsAD', 'Paper_NCvsMCI', 'Paper_All', 'Per_run']
工作表: ['ReadMe', 'Paper_NCvsMCIvsAD', 'Paper_NCvsMCI', 'Paper_All', 'Per_run']
Paper_NCvsMCI 行数: 36  Paper_NCvsMCIvsAD 行数: 54
最终文件: /root/autodl-tmp/ADSST-Net/figs/CAEEG_CAUEEG_decoding_metrics.xlsx
